In [1]:
# 리랭킹적용 - 하이브리드서치와는 다른 고도화 기법 - 1차로 대충 많이 뽑고, 2차로 더 정교하게 다시 순위를 매기는 2단계 검색구조
from langchain_core.documents import Document # 문서그릇
from langchain_community.retrievers import BM25Retriever # 키워드 검색기

# 실습용 사내 IT 규정 문서를 정의합니다.
raw_documents = [
    Document(page_content="보안 규정: 업무용 PC는 3개월마다 비밀번호를 변경해야 하며 특수문자를 포함해야 합니다."),
    Document(page_content="원격 근무 가이드: 재택 근무 시 반드시 사내 VPN을 연결하여 사내 인트라넷에 접속해야 합니다."),
    Document(page_content="장비 지급 규정: 신규 입사자에게는 노트북 1대와 외장 모니터 1대가 기본 지급됩니다."),
    Document(page_content="계정 관리: 사내 계정 5회 연속 로그인 실패 시 계정이 잠기며 IT 지원팀에 잠금 해제를 요청해야 합니다.")
]

# 공백 기준 단순 토크나이저 함수를 정의합니다.
def custom_tokenizer(text: str) -> list[str]:
    # 문장을 띄어쓰기 단위로 쪼개어 단어 목록을 생성합니다.
    return text.split()

# 1단계 기본 검색기(BM25)를 생성하고 후보군 추출 개수를 3개로 설정합니다.
base_retriever = BM25Retriever.from_documents(
    documents=raw_documents,
    preprocess_func=custom_tokenizer
)
# 1단계 검색에서 상위 3개의 문서를 뽑아오도록 지정합니다.
base_retriever.k = 3

C:\Users\kimmi\AppData\Local\Temp\ipykernel_8548\1931242993.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever # 키워드 검색기


In [2]:
# Reranker블록을 만든다.

# 허깅페이스 기반 Cross-Encoder 래퍼 클래스를 가져옵니다.
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
# 문서를 재정렬해 상위 결과를 선별하는 리랭커 모듈을 가져옵니다.
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

# 기본 검색기와 압축기를 하나로 묶어주는 리트리버를 가져옵니다.
from langchain_classic.retrievers import ContextualCompressionRetriever

# 한국어와 영어를 지원하는 다국어 경량 Cross-Encoder 모델을 초기화합니다.
cross_encoder_model = HuggingFaceCrossEncoder(model_name="Dongjin-kr/ko-reranker")

# Cross-Encoder 기반 리랭커를 정의하고, 재정렬 후 최종 2건만 남기도록 지정합니다.
reranker = CrossEncoderReranker(
    model=cross_encoder_model,
    top_n=2
)

# 1차 검색기(base_retriever)와 2차 리랭커(reranker)를 연결합니다.
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [ ]:
from langchain_core.documents import BaseDocumentTransformer
from langchain_community.retrievers import BM25Retriever
from rank_bm25 import BM25Okapi

raw_documents = [
    Document(page_content="보안 규정: 업무용 PC는 3개월마다 비밀번호를 변경해야 하며 특수문자를 포함해야 합니다."),
    Document(page_content="원격 근무 가이드: 재택 근무 시 반드시 사내 VPN을 연결하여 사내 인트라넷에 접속해야 합니다."),
    Document(page_content="장비 지급 규정: 신규 입사자에게는 노트북 1대와 외장 모니터 1대가 기본 지급됩니다."),
    Document(page_content="계정 관리: 사내 계정 5회 연속 로그인 실패 시 계정이 잠기며 IT 지원팀에 잠금 해제를 요청해야 합니다.")
]

def custom_tokenizer(text):
    return text.split()

bm25_retriever = BM25Retriever.from_documents(
    documents=raw_documents,
    preprocess_func=custom_tokenizer, # 전처리함수
)

bm25_retriever.k = 3

In [4]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.retrievers import ContextualCompressionRetriever

cross_encoder_model = HuggingFaceCrossEncoder(model_name="Dongjin-kr/ko-reranker")

reranker = CrossEncoderReranker(
    model=cross_encoder_model,
    top_n=2
)

rerank_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=bm25_retriever
)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [5]:
query = "사내 망 들어가려다 비밀번호 여러 번 틀렸을 때 처리 방법"

print("=== BM25 ===")
bm25_docs = bm25_retriever.invoke(query)
for i, doc in enumerate(bm25_docs, 1):
    print(f"Rank:{i}, Doc:{doc.page_content}")

print("=== ReRanker ===")
rerank_docs = rerank_retriever.invoke(query)
for i, doc in enumerate(rerank_docs, 1):
    print(f"Rank:{i}, Doc:{doc.page_content}")

=== BM25 ===
Rank:1, Doc:계정 관리: 사내 계정 5회 연속 로그인 실패 시 계정이 잠기며 IT 지원팀에 잠금 해제를 요청해야 합니다.
Rank:2, Doc:장비 지급 규정: 신규 입사자에게는 노트북 1대와 외장 모니터 1대가 기본 지급됩니다.
Rank:3, Doc:원격 근무 가이드: 재택 근무 시 반드시 사내 VPN을 연결하여 사내 인트라넷에 접속해야 합니다.
=== ReRanker ===
Rank:1, Doc:계정 관리: 사내 계정 5회 연속 로그인 실패 시 계정이 잠기며 IT 지원팀에 잠금 해제를 요청해야 합니다.
Rank:2, Doc:원격 근무 가이드: 재택 근무 시 반드시 사내 VPN을 연결하여 사내 인트라넷에 접속해야 합니다.
